# NpuKit — int8 matmul on PYNQ-Z2

Loads `npukit.bit` (+ `.hwh` for DMA) and runs:

1. **Classic 8×8** — demo / zeros / identity / int8 extremes / neg×pos / 5× random
2. **Tiled** — larger MxKxN (default 32×32×32) with K accumulation over 8×8 tiles

Keep `npukit.bit`, `npukit.hwh`, and `npukit_matmul.py` in this folder.

In [ ]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_matmul as nk

importlib.reload(nk)

mmio, transport = nk.open_device(BIT)
ident = mmio.read(nk.REG_ID)
assert ident == nk.ID_MAGIC, f"BAD ID 0x{ident:08X}"
print(
    f"ID OK version=0x{mmio.read(nk.REG_VERSION):08X} "
    f"N={mmio.read(nk.REG_N)} transport={type(transport).__name__}"
)

## Classic 8×8 suite

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
classic = nk.classic_8x8_cases(rng)
p0, t0 = nk.run_suite(mmio, transport, classic)
print(f"\nclassic: {p0}/{t0} PASS")
assert p0 == t0

## Tiled suite (software tiling over the 8×8 accelerator)

In [ ]:
M, K, Ndim = 32, 32, 32  # must be multiples of 8
tiled = nk.tiled_cases(M, K, Ndim, rng)
p1, t1 = nk.run_suite(mmio, transport, tiled)
print(f"\ntiled: {p1}/{t1} PASS")
assert p1 == t1
print(f"\nall: {p0 + p1}/{t0 + t1} PASS")

## Optional: one-shot CLI-equivalent

Same as `python3 npukit_matmul.py npukit.bit 32 32 32` (reloads the bitstream).

In [ ]:
%run /home/xilinx/jupyter_notebooks/npukit_matmul.py /home/xilinx/jupyter_notebooks/npukit.bit 32 32 32